### Data Reading

inferSchema ใน PySpark คือการให้ Spark “เดาประเภทข้อมูล (Data Type)” ของแต่ละ column อัตโนมัติ ตอนอ่านไฟล์ เช่น CSV

In [0]:
df = spark.read.csv(
    "/Volumes/apachespark/demodata/raw_data/BigMart Sales.csv",
    header=True,
    inferSchema=True
)

In [0]:
df.show()  
#display (df)

In [0]:
df.printSchema()

#### Data reading JSON

In [0]:
df_json = spark.read.json(
    "/Volumes/apachespark/demodata/raw_data/drivers.json"
)

In [0]:
df_json.printSchema()


In [0]:
display(df_json)


### DDL Schema

ใช้กำหนด Data Type ของ column แบบ string คล้าย SQL แทน `inferSchema`







In [0]:
my_ddl_schema = '''
                    Item_Identifier string,
                    Item_Weight string,
                    Item_Fat_Content string,
                    Item_Visibility double,
                    Item_Type string,
                    Item_MRP double,
                    Outlet_Identifier string,
                    Outlet_Establishment_Year long,
                    Outlet_Size string,
                    Outlet_Location_Type string,
                    Outlet_Type string,
                    Item_Outlet_Sales double
                '''

In [0]:
df = spark.read.format("csv") \
    .schema(my_ddl_schema) \
    .option("header", True) \
    .load("/Volumes/apachespark/demodata/raw_data/BigMart Sales.csv")

In [0]:
display(df)

In [0]:
df.printSchema()

### StructType ()

กำหนด schema แบบ PySpark object ละเอียดกว่า DDL

โครงสร้าง

```python
StructType([
    StructField("column", DataType(), nullable)
])
```



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import functions as F

In [0]:
my_struct_schema = StructType([
    StructField('Item_Identifier', StringType(), True),
    StructField('Item_Weight',DoubleType(), True),
    StructField('Item_Fat_Content', StringType(), True),
    StructField('Item_Visibility', StringType(), True),
    StructField('Item_Type', StringType(), True),
    StructField('Item_MRP', StringType(), True),
    StructField('Outlet_Identifier', StringType(), True),
    StructField('Outlet_Establishment_Year', StringType(), True),
    StructField('Outlet_Size', StringType(), True),
    StructField('Outlet_Location_Type', StringType(), True),
    StructField('Outlet_Type', StringType(), True),
    StructField('Item_Outlet_Sales', StringType(), True),
])
# True refer to allow null in the column

In [0]:
df = spark.read.format("csv") \
    .schema(my_struct_schema) \
    .option("header", True) \
    .load("/Volumes/apachespark/demodata/raw_data/BigMart Sales.csv")

In [0]:
df.printSchema()

### SELECT

In [0]:
df.select('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').limit(20).display(20)


In [0]:
df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).limit(20).display()

### ALIAS

Alias คือ ตั้งชื่อเล่น**ชั่วคราว**ให้ column หรือ table เพื่อให้ใช้ง่ายขึ้น

โครงสร้าง

```python
df.select(col("ชื่อเดิม").alias("ชื่อใหม่"))
```

- ไม่เปลี่ยนชื่อจริง ใช้ได้เฉพาะ query/DataFrame นั้น

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).limit(20).display()

### FILTER and Where 


In [0]:
df = spark.read.csv(
    "/Volumes/apachespark/demodata/raw_data/BigMart Sales.csv",
    header=True,
    inferSchema=True
)

In [0]:
df.filter(col('Item_Fat_Content') == 'Regular').display()

In [0]:
df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight')<10)).display()  

#### isin() , isnull()

In [0]:
df.filter((col("Outlet_Location_Type").isin("Tier 3" ,"Tier 2")) & (col("Outlet_Size").isNull())).display()


#### between()

ใช้กรองค่าที่อยู่ระหว่างช่วงที่กำหนด

In [0]:
df.filter(col('Item_MRP').between(100, 200)).display()

#### contains()

ใช้กรองข้อความที่มีคำที่กำหนดอยู่ข้างใน

In [0]:
df.filter(col('Item_Type').contains('Food')).display()

#### startswith()

ใช้กรองข้อความที่ขึ้นต้นด้วยค่าที่กำหนด

In [0]:
df.filter(col('Item_Identifier').startswith('FD')).display()

#### endswith()

ใช้กรองข้อความที่ลงท้ายด้วยค่าที่กำหนด

In [0]:
df.filter(col('Item_Identifier').endswith('01')).display()

#### like()

ใช้กรองข้อความแบบ pattern คล้าย SQL

In [0]:
df.filter(col('Item_Type').like('%Food%')).display()

### `withColumnRenamed`
เปลี่ยนชื่อคอลัมน์

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

### withColumn()

ใช้สร้าง column ใหม่ หรือแก้ค่า column เดิมใน DataFrame

โครงสร้าง:

```python
df.withColumn("ชื่อคอลัมน์", ค่าใหม่)
```


In [0]:
df = df.withColumn('flag',lit("new"))
#lit() ใช้สร้างค่าคงที่ (constant value) ใส่ใน column

In [0]:
display(df)

#### regexp_replace()

ใช้แทนที่ข้อความใน column

โครงสร้าง

```python
regexp_replace(col("ชื่อคอลัมน์"), "ข้อความเดิม", "ข้อความใหม่")
```

In [0]:
df = df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","Lf"))

df.display()

### Type Casting

In [0]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))  #ใช้เปลี่ยน Data Type ของ column

df.printSchema()

### Sort or OrderBy


ใช้เรียงข้อมูลใน DataFrame

```python
df.sort(["column1","column2"], ascending=[True,False])
```
- asc / True / 1 = น้อย→มาก
- desc / False / 0 = มาก→น้อย

In [0]:
df.sort(desc('Item_Weight')).display() #เรียงจากมากไปน้อย

In [0]:
df.sort(asc('Item_Visibility')).display() #เรียงจากน้อยไปมาก

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending = [0,0]).display()

In [0]:
df.sort(['Item_weight','Item_Visibility'], ascending = [0,1]).display()

#### OrderBy

`orderBy()` ใช้เรียงข้อมูลเหมือน `sort()`

In [0]:
df.orderBy(desc('Item_MRP')).display()

### Limit

In [0]:
df.limit(10).display() 

### SparkSession

ใช้สร้างหรือดึง SparkSession มาใช้งาน

> ใน Databricks ปกติจะมี `spark` ให้ใช้อยู่แล้ว แต่ถ้ารันนอก Databricks อาจต้องสร้างเอง

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Pyspark-Ahhhh').getOrCreate()


### Data Reading Utils

ใช้ดูไฟล์หรือโฟลเดอร์ใน DBFS หรือ Volume

In [0]:
dbutils.fs.ls('/Volumes/apachespark/demodata/raw_data/')

### Basic Actions
Action คือคำสั่งที่สั่งให้ Spark เริ่มประมวลผลจริง

#### count()


In [0]:
df.count()

#### describe()

ใช้ดูค่า statistics of column ที่เป็นตัวเลข เช่น count, mean, stddev, min, max

In [0]:
df.describe().display()

#### head()

ใช้ดึงข้อมูลแถวแรก ๆ ออกมาดู

In [0]:
df.head(5)

#### take()

ใช้ดึงข้อมูลตามจำนวนแถวที่กำหนด คล้าย head()

In [0]:
df.take(5)

#### first()

ใช้ดึงข้อมูลแถวแรกสุด

In [0]:
df.first()

#### toPandas()

แปลง Spark DataFrame เป็น Pandas DataFrame

> ควรใช้กับข้อมูลจำนวนน้อยเท่านั้น เพราะข้อมูลจะถูกดึงมาไว้ที่เครื่อง Driver

In [0]:
df.limit(10).toPandas()

#### toJSON()

แปลงแต่ละ row ให้อยู่ในรูปแบบ JSON string

In [0]:
df.limit(5).toJSON().take(5)

#### cache()

เก็บ DataFrame ไว้ใน memory เพื่อให้เรียกใช้ซ้ำได้เร็วขึ้น

In [0]:
#df_cache = df.cache()

#df_cache.count()

#### explain()

ใช้ดูแผนการทำงานของ Spark ว่าจะประมวลผลยังไง

In [0]:
df.filter(col('Item_Fat_Content') == 'Reg').explain()


### Drop Column

ลบ column ที่ไม่ต้องการออกจาก DataFrame

In [0]:
df.drop('flag').display()

### Distinct and Duplicates

ใช้จัดการค่าซ้ำใน DataFrame

#### distinct()

ใช้เอาค่าที่ไม่ซ้ำออกมา

In [0]:
df.distinct().display()

In [0]:
df.select('Item_Fat_Content').distinct().display()

#### dropDuplicates()

ใช้ลบแถวที่ซ้ำกัน โดยกำหนด column ที่ต้องการเช็คได้

In [0]:
df.dropDuplicates().display()

In [0]:
df.drop_duplicates(subset=['Item_Type']).display()
#df.drop_duplicates(['Item_Type']).display()
#df.dropDuplicates(['Item_Type']).display()

### String Functions

ใช้จัดการข้อความใน column

#### upper()

เปลี่ยนข้อความให้เป็นตัวพิมพ์ใหญ่ทั้งหมด

In [0]:
df.select(upper('Item_Type').alias('upper_Item_Type')).display()

#### lower()

เปลี่ยนข้อความให้เป็นตัวพิมพ์เล็กทั้งหมด

In [0]:
df.select(lower('Item_Type').alias('lower_Item_Type')).display()

#### substr()

ตัดข้อความจาก column ตามตำแหน่งที่กำหนด

In [0]:
df.select(col('Item_Identifier').substr(1, 2).alias('Item_Code')).display()

### Date Functions

ใช้สร้างและจัดการข้อมูลวันที่

#### current_date()

สร้าง column วันที่ปัจจุบัน

In [0]:
df = df.withColumn('curr_date', current_date())

df.display()

#### date_add()

บวกจำนวนวันเพิ่มจากวันที่เดิม

In [0]:
 df = df.withColumn('week_after', date_add('curr_date', 7))

df.display()

#### date_sub()

ลบจำนวนวันจากวันที่เดิม

In [0]:
df = df.withColumn('week_before', date_sub('curr_date', 7))

df.display()

#### datediff()

หาความต่างของวันที่ หน่วยเป็นจำนวนวัน

In [0]:
  df = df.withColumn('datediff', datediff('week_after', 'curr_date'))

df.display()

#### date_format()

เปลี่ยนรูปแบบการแสดงผลวันที่

In [0]:
df = df.withColumn('week_before_format', date_format('week_before', 'dd-MM-yyyy'))

df.display()

### Handling Nulls

ใช้จัดการข้อมูลที่เป็นค่าว่าง หรือ `null`

#### dropna()

ใช้ลบแถวที่มีค่า null


In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_Size']).display()

#### fillna()

ใช้เติมค่าลงในช่องที่เป็น null

In [0]:
df.fillna('NotAvailable').display()

In [0]:
df.fillna('NotAvailable', subset=['Outlet_Size']).display()

### Split and Array Functions

ใช้แยกข้อความให้กลายเป็น array และจัดการข้อมูลแบบ array

#### split()

ใช้แยกข้อความออกเป็นหลายส่วน

In [0]:
df.withColumn('Outlet_Type_Split', split('Outlet_Type', ' ')).display()

#### Indexing

ใช้ดึงข้อมูลบางตำแหน่งจาก array

In [0]:
df.withColumn('Outlet_Type_Index', split('Outlet_Type', ' ')[1]).display()

#### explode()

ใช้แตก array ให้กลายเป็นหลาย row

In [0]:
df_exp = df.withColumn('Outlet_Type_Array', split('Outlet_Type', ' '))

df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type_Explode', explode('Outlet_Type_Array')).display()

#### array_contains()

ใช้เช็คว่าใน array มีค่าที่ต้องการอยู่ไหม

In [0]:
df_exp.withColumn('Type1_flag', array_contains('Outlet_Type_Array', 'Type1')).display()

### GroupBy and Aggregation

ใช้จัดกลุ่มข้อมูล แล้วคำนวณค่าสรุป

#### groupBy() + sum()

รวมค่าตามกลุ่มที่กำหนด

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

#### groupBy() + avg()

หาค่าเฉลี่ยตามกลุ่มที่กำหนด

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).display()

#### agg() หลาย column

ใช้คำนวณหลายค่าในครั้งเดียว

In [0]:
df.groupBy('Item_Type', 'Outlet_Size').agg(
    sum('Item_MRP').alias('Total_MRP'),
    avg('Item_MRP').alias('Avg_MRP')
).display()

#### collect_list()

รวมค่าหลาย row ให้อยู่ใน list เดียวกัน

In [0]:
df_collect = df.select('Outlet_Identifier', 'Item_Type') \
    .dropna(subset=['Outlet_Identifier', 'Item_Type'])

df_collect.display()


In [0]:
df_collect.groupBy('Outlet_Identifier') \
    .agg(collect_list('Item_Type').alias('Item_Type_List')) \
    .display()


### Pivot

ใช้หมุนค่าใน column ให้กลายเป็น column ใหม่

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

### When-Otherwise

ใช้สร้างเงื่อนไขคล้าย `if-else`

In [0]:
df = df.withColumn(
    'veg_flag',
    when(col('Item_Type') == 'Meat', 'Non-Veg').otherwise('Veg')
)

df.display()

In [0]:
df.withColumn(
    'veg_exp_flag',
    when(((col('veg_flag') == 'Veg') & (col('Item_MRP') < 100)), 'Veg_Inexpensive')
    .when(((col('veg_flag') == 'Veg') & (col('Item_MRP') >= 100)), 'Veg_Expensive')
    .otherwise('Non_Veg')
).display()

### Joins

ใช้รวมข้อมูลจาก DataFrame 2 ตัวเข้าด้วยกัน

ตัวอย่างนี้จะแยกข้อมูลจาก BigMart เป็น 2 DataFrame ก่อน แล้วค่อย join กลับด้วย `Item_Identifier`


In [0]:
df_item = df.select(
    'Item_Identifier',
    'Item_Type',
    'Item_Fat_Content',
    'Item_MRP'
).dropDuplicates(['Item_Identifier'])

df_outlet = df.select(
    'Item_Identifier',
    'Outlet_Identifier',
    'Outlet_Type',
    'Outlet_Size'
).dropDuplicates(['Item_Identifier'])

# ใช้สำหรับตัวอย่าง left/anti join ให้เห็นแถวที่ไม่มีคู่ match ชัดขึ้น
df_outlet_notnull = df_outlet.filter(col('Outlet_Size').isNotNull())


In [0]:
df_item.display()


In [0]:
df_outlet.display()

#### Inner Join

เอาเฉพาะข้อมูลที่ match กันทั้ง 2 ฝั่ง

In [0]:
df_item.join(df_outlet, 'Item_Identifier', 'inner').display()


#### Left Join

เอาข้อมูลฝั่งซ้ายทั้งหมด และดึงข้อมูลฝั่งขวามาต่อถ้ามีค่าที่ match กัน

In [0]:
df_item.join(df_outlet_notnull, 'Item_Identifier', 'left').display()


#### Right Join

เอาข้อมูลฝั่งขวาทั้งหมด และดึงข้อมูลฝั่งซ้ายมาต่อถ้ามีค่าที่ match กัน

In [0]:
df_item.join(df_outlet, 'Item_Identifier', 'right').display()


#### Anti Join

เอาข้อมูลฝั่งซ้ายที่ไม่มีคู่ match กับฝั่งขวา

In [0]:
df_item.join(df_outlet_notnull, 'Item_Identifier', 'anti').display()


### Union

ใช้ต่อข้อมูลหลาย DataFrame เข้าด้วยกันในแนว row

ตัวอย่างนี้ใช้แยกสินค้าคนละกลุ่มออกมา แล้วนำมาต่อกัน



In [0]:
df_food = df.select('Item_Identifier', 'Item_Type') \
    .filter(col('Item_Identifier').startswith('FD')) \
    .limit(5)

df_drink = df.select('Item_Identifier', 'Item_Type') \
    .filter(col('Item_Identifier').startswith('DR')) \
    .limit(5)


In [0]:
df_food.union(df_drink).display()


#### unionByName()

ใช้ union โดยอิงจากชื่อ column ไม่ได้อิงจากลำดับ column

In [0]:
df_drink_reverse = df_drink.select('Item_Type', 'Item_Identifier')

df_drink_reverse.display()


In [0]:
df_drink_reverse.unionByName(df_food).display()


### Window Functions

ใช้คำนวณข้ามแถว โดยยังไม่ยุบ row เหมือน `groupBy()`

In [0]:
from pyspark.sql.window import Window

#### row_number()

ใส่เลขลำดับให้แต่ละแถว

In [0]:
df.withColumn(
    'rowCol',
    row_number().over(Window.orderBy('Item_Identifier'))
).display()

#### rank() and dense_rank()

ใช้จัดอันดับข้อมูล

- `rank()` ถ้ามีอันดับซ้ำ จะเว้นเลขอันดับถัดไป
- `dense_rank()` ถ้ามีอันดับซ้ำ จะไม่เว้นเลขอันดับถัดไป

In [0]:
df.withColumn(
    'rank',
    rank().over(Window.orderBy(col('Item_Identifier').desc()))
).withColumn(
    'denseRank',
    dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))
).display()

#### rowsBetween()

กำหนดช่วงแถวที่ต้องการให้ Window Function คำนวณ

In [0]:
df.withColumn(
    'cumsum',
    sum('Item_MRP').over(
        Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
).display()

In [0]:
df.withColumn(
    'totalsum',
    sum('Item_MRP').over(
        Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
    )
).display()

### User Defined Functions (UDF)

ใช้สร้างฟังก์ชันเอง แล้วเอาไปใช้กับ column ใน Spark DataFrame

#### STEP 1 สร้าง Python function ปกติก่อน

In [0]:
def my_func(x):
    if x is not None:
        return x * x
    return None

#### STEP 2 แปลง Python function ให้เป็น UDF ของ Spark

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('mynewcol', my_udf('Item_MRP')).display()


### Data Writing

ใช้เขียน DataFrame ออกไปเป็นไฟล์หรือ table

> เปลี่ยน output_path ให้ตรงกับ path ที่ต้องการเก็บไฟล์

In [0]:
output_path = "/Volumes/apachespark/demodata/raw_data/BigMart_Sales_Output"


#### save()

เขียน DataFrame ออกไปตาม format ที่กำหนด

In [0]:
#df.write.format('csv').save(output_path)

#### mode('append')

เขียนข้อมูลต่อท้ายไฟล์เดิม


In [0]:
df.write.format('csv')     .mode('append')     .save(output_path)

#### mode('overwrite')

เขียนทับข้อมูลเดิม

In [0]:
df.write.format('csv')     .mode('overwrite')     .save(output_path)

#### mode('ignore')

ถ้ามี path อยู่แล้วจะไม่เขียนทับ และไม่ error

In [0]:
df.write.format('csv')     .mode('ignore')     .save(output_path)

#### mode('error')

ถ้ามี path อยู่แล้วจะ error

In [0]:
#df.write.format('csv')     .mode('error')     .save(output_path)

#### parquet()

เขียนข้อมูลเป็นไฟล์ Parquet

In [0]:
parquet_path = "/Volumes/apachespark/demodata/raw_data/bigmart_parquet"

df.write \
    .mode("overwrite") \
    .parquet(parquet_path)

In [0]:
df_parquet = spark.read.parquet(parquet_path)

df_parquet.display()

#### saveAsTable()

เขียน DataFrame เป็น table

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bigmart_table")

### Spark SQL

ใช้เขียน SQL query กับ DataFrame ได้ โดยต้องสร้าง temp view ก่อน

#### createTempView()

สร้าง view ชั่วคราวจาก DataFrame

In [0]:
df.createTempView('bigmart_view')

In [0]:
%sql

select *
from bigmart_view
where Item_Fat_Content = 'Lf'

In [0]:
df_sql = spark.sql("""
    select *
    from bigmart_view
    where Item_Fat_Content = 'Lf'
""")

df_sql.display()

#### createOrReplaceTempView()

สร้าง view ใหม่ หรือแทนที่ view เดิมถ้ามีชื่อซ้ำ

In [0]:
df.createOrReplaceTempView('bigmart_view_replace')

In [0]:
df_sql_replace = spark.sql("""
    select *
    from bigmart_view_replace
    where Item_Fat_Content = 'Lf'
""")

df_sql_replace.display()